In [ ]:
import os
from pathlib import Path

# Force a stable W&B directory across sessions/devices
WANDB_DIR = (Path.cwd().resolve() / "experiments" / "artifacts" / "wandb")
os.environ["WANDB_DIR"] = str(WANDB_DIR)
WANDB_DIR.mkdir(parents=True, exist_ok=True)


# Mistral-7B (Unsloth LoRA) - Abstract Evaluator SFT

This notebook trains `mistralai/Mistral-7B-Instruct-v0.3` using Unsloth + LoRA on chat-format JSONL data, logs to Weights & Biases, evaluates each epoch, and reports ROUGE/BLEU/BERTScore on generated rationales plus precision/recall/F1/accuracy on predicted scores.

In [ ]:
# If running first time, uncomment:
# !pip install -U unsloth transformers datasets trl peft accelerate bitsandbytes wandb evaluate rouge_score nltk bert_score scikit-learn

import os
import json
import random
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, DatasetDict

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ------------------------------
# Paths and experiment config
# ------------------------------
PROJECT_ROOT = r"C:\\Users\\hanib\\Desktop\\nlp_final_final_final_final_project\\Abstract-Evaluator"
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'sft')
os.makedirs(DATA_DIR, exist_ok=True)

# Preferred: provide these JSONL files directly
TRAIN_PATH = os.path.join(DATA_DIR, 'train.jsonl')
VAL_PATH = os.path.join(DATA_DIR, 'val.jsonl')
TEST_PATH = os.path.join(DATA_DIR, 'test.jsonl')

# Optional fallback: one combined JSONL with exactly 10k rows
COMBINED_PATH = os.path.join(DATA_DIR, 'all.jsonl')

MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.3'
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'mistral7b_unsloth_lora')
RUN_NAME = 'mistral7b-abstract-evaluator-unsloth-lora'

# Sequence and LoRA setup
MAX_SEQ_LENGTH = 1024
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

# Mistral-7B has 32 transformer layers
# Train top 12 layers with LoRA -> freeze 20/32 = 62.5% (within your 50-70% target)
TOTAL_LAYERS = 32
TRAIN_TOP_K = 12
LAYERS_TO_TRANSFORM = list(range(TOTAL_LAYERS - TRAIN_TOP_K, TOTAL_LAYERS))

# For ~80GB VRAM this is usually safe with 4-bit + gradient checkpointing
PER_DEVICE_TRAIN_BATCH_SIZE = 16
PER_DEVICE_EVAL_BATCH_SIZE = 16
GRAD_ACC_STEPS = 2
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 3
WARMUP_RATIO = 0.03

print('Will LoRA-train layers:', LAYERS_TO_TRANSFORM)

In [ ]:
# ------------------------------
# Dataset loading + split checks
# ------------------------------
def file_exists(p):
    return os.path.isfile(p) and os.path.getsize(p) > 0

if file_exists(TRAIN_PATH) and file_exists(VAL_PATH) and file_exists(TEST_PATH):
    data_files = {'train': TRAIN_PATH, 'validation': VAL_PATH, 'test': TEST_PATH}
    ds = load_dataset('json', data_files=data_files)
else:
    assert file_exists(COMBINED_PATH), (
        f'Missing split files and missing combined file: {COMBINED_PATH}. '
        'Provide train/val/test JSONL or all.jsonl in data/sft.'
    )
    full = load_dataset('json', data_files={'all': COMBINED_PATH})['all'].shuffle(seed=SEED)
    assert len(full) >= 10000, f'Need at least 10,000 rows, found {len(full)}'
    full = full.select(range(10000))
    ds = DatasetDict({
        'train': full.select(range(0, 8000)),
        'validation': full.select(range(8000, 9000)),
        'test': full.select(range(9000, 10000)),
    })

print(ds)
print('Train size:', len(ds['train']))
print('Val size:', len(ds['validation']))
print('Test size:', len(ds['test']))

# Basic format check
sample = ds['train'][0]
assert 'messages' in sample, 'Each row must contain `messages`'
assert isinstance(sample['messages'], list), '`messages` must be a list'
assert len(sample['messages']) >= 2, 'Need at least user + assistant messages'
print('Sample OK:', sample['messages'][0]['role'], '->', sample['messages'][1]['role'])

In [ ]:
# ------------------------------
# Weights & Biases
# ------------------------------
import wandb

# Option 1: set WANDB_API_KEY in environment before running
# Option 2: run wandb.login(key='...')
wandb.login()

wandb.init(
    project='abstract-evaluator-sft',
    name=RUN_NAME,
    config={
        'model': MODEL_NAME,
        'max_seq_length': MAX_SEQ_LENGTH,
        'lora_r': LORA_R,
        'lora_alpha': LORA_ALPHA,
        'lora_dropout': LORA_DROPOUT,
        'layers_to_transform': LAYERS_TO_TRANSFORM,
        'batch_size': PER_DEVICE_TRAIN_BATCH_SIZE,
        'grad_accumulation': GRAD_ACC_STEPS,
        'learning_rate': LEARNING_RATE,
        'epochs': NUM_EPOCHS,
        'train_rows': len(ds['train']),
        'val_rows': len(ds['validation']),
        'test_rows': len(ds['test'])
    }
)

In [ ]:
# ------------------------------
# Model + tokenizer (Unsloth)
# ------------------------------
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth',
    bias='none',
    random_state=SEED,
    layers_to_transform=LAYERS_TO_TRANSFORM,
)

print('LoRA model prepared.')

In [ ]:
# ------------------------------
# Formatting for chat SFT
# ------------------------------
def formatting_prompts_func(examples):
    texts = []
    for msgs in examples['messages']:
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {'text': texts}

train_ds = ds['train'].map(formatting_prompts_func, batched=True)
val_ds = ds['validation'].map(formatting_prompts_func, batched=True)
test_ds = ds['test'].map(formatting_prompts_func, batched=True)

print(train_ds[0]['text'][:800])

In [ ]:
# ------------------------------
# Trainer setup
# ------------------------------
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

os.makedirs(OUTPUT_DIR, exist_ok=True)

train_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    run_name=RUN_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=NUM_EPOCHS,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    logging_steps=10,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to=['wandb'],
    bf16=torch.cuda.is_available(),
    fp16=not torch.cuda.is_available(),
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field='text',
    args=train_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

trainer

In [ ]:
# ------------------------------
# Train
# ------------------------------
train_result = trainer.train()
print(train_result)

best_ckpt = trainer.state.best_model_checkpoint
print('Best checkpoint:', best_ckpt)

trainer.save_model(os.path.join(OUTPUT_DIR, 'final_lora_adapter'))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, 'final_lora_adapter'))

In [ ]:
# ------------------------------
# Validation and test loss
# ------------------------------
val_metrics = trainer.evaluate(eval_dataset=val_ds)
test_loss_metrics = trainer.evaluate(eval_dataset=test_ds, metric_key_prefix='test')
print('Validation metrics:', val_metrics)
print('Test loss metrics:', test_loss_metrics)
wandb.log({**val_metrics, **test_loss_metrics})

In [ ]:
# ------------------------------
# Full evaluator: generation + rationale metrics + score metrics
# ------------------------------
# This cell evaluates the final model on the TEST set.
#
# Text metrics on rationale:
#   - ROUGE
#   - BLEU
#   - BERTScore
#
# Score metrics:
#   - precision
#   - recall
#   - F1
#   - accuracy
#
# Expected output format from the assistant can be JSON, for example:
# {"score": 4, "rationale": "..."}
# The parser below is tolerant and also works with text like:
# Score: 4
# Rationale: ...

import re
import json
import math
from typing import Any, Dict, List, Optional, Tuple

import evaluate
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

FastLanguageModel.for_inference(model)

GEN_MAX_NEW_TOKENS = 256
GEN_TEMPERATURE = 0.0
GEN_DO_SAMPLE = False
EVAL_LIMIT = None  # set to a small number like 50 for a quick smoke test

def get_assistant_reference(messages: List[Dict[str, str]]) -> str:
    """Return the last assistant message as the gold/reference answer."""
    for msg in reversed(messages):
        if msg.get("role") == "assistant":
            return str(msg.get("content", ""))
    return ""

def get_prompt_messages(messages: List[Dict[str, str]]) -> List[Dict[str, str]]:
    """Remove the gold assistant response from messages before generation."""
    prompt_messages = []
    removed_gold = False
    for msg in reversed(messages):
        if msg.get("role") == "assistant" and not removed_gold:
            removed_gold = True
            continue
        prompt_messages.insert(0, msg)
    return prompt_messages

def safe_json_loads_maybe(text: str) -> Optional[Dict[str, Any]]:
    """Try to parse a JSON object from model/reference text."""
    if not text:
        return None

    cleaned = text.strip()

    # Remove markdown fences if present.
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    candidates = [cleaned]

    # Try the first {...} block.
    match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
    if match:
        candidates.append(match.group(0))

    for cand in candidates:
        try:
            obj = json.loads(cand)
            if isinstance(obj, dict):
                return obj
        except Exception:
            pass

    return None

def normalize_score(value: Any) -> Optional[int]:
    """Convert score into an integer class when possible."""
    if value is None:
        return None

    if isinstance(value, (int, np.integer)):
        return int(value)

    if isinstance(value, (float, np.floating)) and not math.isnan(float(value)):
        return int(round(float(value)))

    text = str(value).strip()

    # Handles examples like "3: reject, not good enough" or "score = 3"
    match = re.search(r"-?\d+(?:\.\d+)?", text)
    if match:
        return int(round(float(match.group(0))))

    return None

def parse_score_and_rationale(text: str) -> Dict[str, Any]:
    """Extract score and rationale from JSON or semi-structured text."""
    obj = safe_json_loads_maybe(text)

    if obj is not None:
        score = None
        rationale = None

        # Common score key names
        for key in ["score", "rating", "grade", "predicted_score"]:
            if key in obj:
                score = normalize_score(obj.get(key))
                break

        # Common rationale key names
        for key in ["rationale", "rational", "reason", "reasoning", "explanation", "feedback"]:
            if key in obj and obj.get(key) is not None:
                rationale = str(obj.get(key)).strip()
                break

        # If rationale is nested/list, fallback to full JSON string.
        if rationale is None:
            rationale = json.dumps(obj, ensure_ascii=False)

        return {"score": score, "rationale": rationale}

    # Non-JSON fallback.
    score = None
    score_match = re.search(r"(?:score|rating|grade)\s*[:=]\s*(-?\d+(?:\.\d+)?)", text, flags=re.IGNORECASE)
    if score_match:
        score = normalize_score(score_match.group(1))
    else:
        score = normalize_score(text)

    rationale = text.strip()

    rationale_match = re.search(
        r"(?:rationale|rational|reason|reasoning|explanation|feedback)\s*[:=]\s*(.*)",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )
    if rationale_match:
        rationale = rationale_match.group(1).strip()

    return {"score": score, "rationale": rationale}

def generate_one(messages: List[Dict[str, str]]) -> str:
    """Generate the model answer for one chat example."""
    prompt_messages = get_prompt_messages(messages)

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=GEN_MAX_NEW_TOKENS,
            do_sample=GEN_DO_SAMPLE,
            temperature=GEN_TEMPERATURE,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

# Build references and predictions.
raw_test = ds["test"]
if EVAL_LIMIT is not None:
    raw_test = raw_test.select(range(min(EVAL_LIMIT, len(raw_test))))

records = []

for idx, row in enumerate(raw_test):
    messages = row["messages"]

    gold_text = get_assistant_reference(messages)
    pred_text = generate_one(messages)

    gold_parsed = parse_score_and_rationale(gold_text)
    pred_parsed = parse_score_and_rationale(pred_text)

    records.append({
        "idx": idx,
        "gold_text": gold_text,
        "pred_text": pred_text,
        "gold_score": gold_parsed["score"],
        "pred_score": pred_parsed["score"],
        "gold_rationale": gold_parsed["rationale"],
        "pred_rationale": pred_parsed["rationale"],
    })

    if (idx + 1) % 25 == 0:
        print(f"Generated {idx + 1}/{len(raw_test)} examples")

eval_df = pd.DataFrame(records)

# Keep text metrics safe when empty strings appear.
eval_df["gold_rationale"] = eval_df["gold_rationale"].fillna("").astype(str)
eval_df["pred_rationale"] = eval_df["pred_rationale"].fillna("").astype(str)

pred_rationales = eval_df["pred_rationale"].tolist()
gold_rationales = eval_df["gold_rationale"].tolist()

# ------------------------------
# Rationale metrics
# ------------------------------
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")
bertscore_metric = evaluate.load("bertscore")

rouge_scores = rouge_metric.compute(
    predictions=pred_rationales,
    references=gold_rationales,
    use_stemmer=True,
)

bleu_scores = bleu_metric.compute(
    predictions=pred_rationales,
    references=[[r] for r in gold_rationales],
)

bertscore_scores = bertscore_metric.compute(
    predictions=pred_rationales,
    references=gold_rationales,
    lang="en",
)

# BERTScore returns one score per example, so average it.
bertscore_summary = {
    "bertscore_precision": float(np.mean(bertscore_scores["precision"])),
    "bertscore_recall": float(np.mean(bertscore_scores["recall"])),
    "bertscore_f1": float(np.mean(bertscore_scores["f1"])),
}

# ------------------------------
# Score classification metrics
# ------------------------------
score_eval_df = eval_df.dropna(subset=["gold_score", "pred_score"]).copy()
score_eval_df["gold_score"] = score_eval_df["gold_score"].astype(int)
score_eval_df["pred_score"] = score_eval_df["pred_score"].astype(int)

if len(score_eval_df) > 0:
    y_true = score_eval_df["gold_score"].tolist()
    y_pred = score_eval_df["pred_score"].tolist()

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0,
    )

    score_metrics = {
        "score_accuracy": float(accuracy_score(y_true, y_pred)),
        "score_precision_macro": float(precision_macro),
        "score_recall_macro": float(recall_macro),
        "score_f1_macro": float(f1_macro),
        "score_precision_weighted": float(precision_weighted),
        "score_recall_weighted": float(recall_weighted),
        "score_f1_weighted": float(f1_weighted),
        "score_examples_evaluated": int(len(score_eval_df)),
        "score_examples_missing_parse": int(len(eval_df) - len(score_eval_df)),
    }

    print("\nClassification report:")
    print(classification_report(y_true, y_pred, zero_division=0))
else:
    score_metrics = {
        "score_examples_evaluated": 0,
        "score_examples_missing_parse": int(len(eval_df)),
    }
    print("No score metrics computed because score parsing failed for all examples.")

# Merge all generation/evaluation metrics.
gen_metrics = {}
gen_metrics.update({f"rouge_{k}": float(v) for k, v in rouge_scores.items()})
gen_metrics.update({"bleu": float(bleu_scores["bleu"])})
gen_metrics.update(bertscore_summary)
gen_metrics.update(score_metrics)

print("\nFinal generation metrics:")
print(json.dumps(gen_metrics, indent=2))

# Save predictions and generation metrics.
os.makedirs(OUTPUT_DIR, exist_ok=True)

predictions_path = os.path.join(OUTPUT_DIR, "test_predictions_with_eval.csv")
gen_metrics_path = os.path.join(OUTPUT_DIR, "generation_eval_metrics.json")

eval_df.to_csv(predictions_path, index=False, encoding="utf-8")

with open(gen_metrics_path, "w", encoding="utf-8") as f:
    json.dump(gen_metrics, f, indent=2)

print("Saved predictions to:", predictions_path)
print("Saved generation metrics to:", gen_metrics_path)

# Log to W&B if available.
try:
    wandb.log(gen_metrics)
    wandb.log({"test_predictions": wandb.Table(dataframe=eval_df)})
except Exception as e:
    print("W&B logging skipped:", e)

In [ ]:
# ------------------------------
# Save metrics + finish W&B
# ------------------------------
metrics_path = os.path.join(OUTPUT_DIR, 'final_metrics.json')
all_metrics = {}

if 'val_metrics' in globals():
    all_metrics.update({k: float(v) for k, v in val_metrics.items() if isinstance(v, (int, float))})

if 'test_loss_metrics' in globals():
    all_metrics.update({k: float(v) for k, v in test_loss_metrics.items() if isinstance(v, (int, float))})

if 'gen_metrics' in globals():
    all_metrics.update(gen_metrics)

with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(all_metrics, f, indent=2)

print('Saved final metrics to:', metrics_path)

try:
    wandb.finish()
except Exception as e:
    print('W&B finish skipped:', e)